# Model 3: MobileNetV2 for Automatic Waste Classification

**Module**: SE4050 – Deep Learning (2026)  
**Author**: S. S. Kumbukage (IT23155534)  
**Track**: Supervised Deep Learning (Waste Classification)  

---

## Step 1: Environment Setup & High-Performance Data Pipeline
In this initial stage, we establish:
1. **Reproducibility Configuration**: Strict random seeds ($42$) across Python, NumPy, and TensorFlow.
2. **Leakage-Safe Data Ingestion**: Loading pre-split manifests (`train.csv`, `validation.csv`, `test.csv`).
3. **Optimized Input Pipeline (`tf.data`)**:
   - Uniform input resolution: $224 \times 224 \times 3$.
   - Batch size: $32$.
   - Normalization via `tf.keras.applications.mobilenet_v2.preprocess_input` ($[-1, 1]$ range).
   - On-the-fly training data augmentation.
   - Pre-fetching and multi-threaded decoding.
4. **Single-Batch Verification**: Ingest and visualize one complete batch to verify tensor shapes and label mapping.


In [4]:
# 1. Core Libraries and Environment Verification
import os
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Hardware Acceleration: GPU Detected ({gpus[0].name})")
else:
    print("Hardware Acceleration: Running in CPU Mode")


TensorFlow Version: 2.18.0
Hardware Acceleration: Running in CPU Mode


In [5]:
# 2. Reproducibility Configuration & Directory Setup
RANDOM_SEED = 42

os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Dynamic Project Root Resolution
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_IMAGES_DIR = DATA_DIR / "raw" / "Garbage_Dataset_Classification" / "images"

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
TABLES_DIR = RESULTS_DIR / "tables"

for d in [FIGURES_DIR, METRICS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Processed Splits  : {PROCESSED_DIR}")
print(f"Raw Images Folder : {RAW_IMAGES_DIR}")
print(f"Random Seed       : {RANDOM_SEED}")


Project Root      : c:\Users\sadee\OneDrive\Documents\SLIIT\Forth Year\First Semester\Deep Learning\Assignment\SE4050-Automatic-Waste-Classification
Processed Splits  : c:\Users\sadee\OneDrive\Documents\SLIIT\Forth Year\First Semester\Deep Learning\Assignment\SE4050-Automatic-Waste-Classification\data\processed
Raw Images Folder : c:\Users\sadee\OneDrive\Documents\SLIIT\Forth Year\First Semester\Deep Learning\Assignment\SE4050-Automatic-Waste-Classification\data\raw\Garbage_Dataset_Classification\images
Random Seed       : 42


In [6]:
# 3. Load Preprocessed Data Manifests (train.csv, validation.csv, test.csv)
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

# Map relative paths to absolute file paths
train_df["image_path"] = train_df["relative_path"].apply(lambda p: str(RAW_IMAGES_DIR / p))
val_df["image_path"] = val_df["relative_path"].apply(lambda p: str(RAW_IMAGES_DIR / p))
test_df["image_path"] = test_df["relative_path"].apply(lambda p: str(RAW_IMAGES_DIR / p))

CLASSES = sorted(train_df["class"].unique())
NUM_CLASSES = len(CLASSES)
LABEL_TO_CLASS = {idx: cls_name for idx, cls_name in enumerate(CLASSES)}
CLASS_TO_LABEL = {cls_name: idx for idx, cls_name in enumerate(CLASSES)}

total_images = len(train_df) + len(val_df) + len(test_df)
print(f"Classes ({NUM_CLASSES}): {CLASSES}")
print(f"Dataset Split Summary:")
print(f"  Training Images   : {len(train_df):,} ({len(train_df)/total_images*100:.1f}%)")
print(f"  Validation Images : {len(val_df):,} ({len(val_df)/total_images*100:.1f}%)")
print(f"  Test Images       : {len(test_df):,} ({len(test_df)/total_images*100:.1f}%)")
print(f"  Total Images      : {total_images:,}")


Classes (6): ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Dataset Split Summary:
  Training Images   : 9,692 (69.7%)
  Validation Images : 2,114 (15.2%)
  Test Images       : 2,091 (15.0%)
  Total Images      : 13,897


In [7]:
# 4. Class Distribution Across Splits
distribution_df = pd.DataFrame({
    'Train': train_df['class'].value_counts(),
    'Validation': val_df['class'].value_counts(),
    'Test': test_df['class'].value_counts()
}).loc[CLASSES]

distribution_df['Total'] = distribution_df.sum(axis=1)
print("Class Distribution Breakdown:")
print(distribution_df)


Class Distribution Breakdown:
           Train  Validation  Test  Total
class                                    
cardboard   1548         322   344   2214
glass       1751         380   367   2498
metal       1440         326   317   2083
paper       1605         364   346   2315
plastic     1598         347   342   2287
trash       1750         375   375   2500


In [8]:
# 5. Build tf.data Input Pipeline with Augmentation and Preprocessing
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Data augmentation layer (applied during training only)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1)
], name="data_augmentation")

def load_and_preprocess_image(path, label):
    """Loads, decodes, resizes, and scales image to [-1, 1] using MobileNetV2 specification."""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img, label

def create_pipeline(df, is_training=False):
    """Constructs an optimized tf.data.Dataset."""
    paths = df["image_path"].values
    labels = df["label"].values
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    
    if is_training:
        ds = ds.shuffle(buffer_size=len(df), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    
    if is_training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
        
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = create_pipeline(train_df, is_training=True)
val_ds = create_pipeline(val_df, is_training=False)
test_ds = create_pipeline(test_df, is_training=False)

print(f"Pipeline Created:")
print(f"  Train Batches      : {len(train_ds)} batches (size {BATCH_SIZE})")
print(f"  Validation Batches : {len(val_ds)} batches (size {BATCH_SIZE})")
print(f"  Test Batches       : {len(test_ds)} batches (size {BATCH_SIZE})")


Pipeline Created:
  Train Batches      : 303 batches (size 32)
  Validation Batches : 67 batches (size 32)
  Test Batches       : 66 batches (size 32)


In [ ]:
# 6. Single Batch Verification & Visualization
batch_images, batch_labels = next(iter(train_ds))

print("Batch Verification Results ---")
print(f"Batch Image Tensor Shape : {batch_images.shape}")
print(f"Batch Labels Tensor Shape: {batch_labels.shape}")
print(f"Image Data Type          : {batch_images.dtype}")
print(f"Label Data Type          : {batch_labels.dtype}")
print(f"Pixel Value Min / Max    : {tf.reduce_min(batch_images):.2f} / {tf.reduce_max(batch_images):.2f} (Expected: [-1.0, 1.0])")

# Denormalize [-1, 1] back to [0, 1] for matplotlib visualization
vis_images = (batch_images.numpy() + 1.0) / 2.0

plt.figure(figsize=(14, 7))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(np.clip(vis_images[i], 0.0, 1.0))
    label_idx = batch_labels[i].numpy()
    plt.title(f"Label: {label_idx} ({LABEL_TO_CLASS[label_idx]})", fontsize=11, fontweight='bold')
    plt.axis('off')

plt.suptitle("Sample Images from Verified Training Batch (with Data Augmentation)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Step 1 Complete: Data pipeline, splits, and tensor ingestion fully verified!")


SyntaxError: unterminated string literal (detected at line 4) (2887709093.py, line 4)